# Notebook 6: Extended Experiment (all improvements in one notebook)

This notebook **re-runs the whole experiment** with the additions that make the paper stronger.
It does not reuse the old result files, because the new measures (balanced accuracy, macro-F1) need the models to be trained again.

**What is new compared with Notebook 4**

1. **Two more performance measures:** balanced accuracy and macro-F1, next to plain accuracy. This shows whether class imbalance really is harmless.
2. **A fifth defect: class-dependent label noise.** Labels are not flipped to a random class but to a fixed neighbouring class, which is more like real noise. The meta-model is trained on symmetric noise only and tested on this new noise (a transfer test).
3. **Two more model families:** LightGBM and a small neural network (MLP), so the result covers 5 models.
4. **Stronger statistics:** confidence intervals (bootstrap over datasets) and Holm correction for multiple tests. You can also add more seeds (see `SEEDS`).
5. **A failure analysis:** which datasets does the meta-model fail on, and what do they have in common?
6. **The "told the noise level" baseline** is now part of the main planned-test table.

**Running time:** much longer than before (about 2x the number of models and 4 more settings). Results are saved per dataset and seed, so you can stop and continue later.
Set `FAST_TEST = True` first to check that everything works (a few minutes).

In [ ]:
!pip install -q openml lightgbm

## Setup: where results are saved

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/data_quality_paper_extended'
except ImportError:
    OUTPUT_DIR = 'results_extended'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Saving results to:', OUTPUT_DIR)

In [ ]:
import time, glob, warnings
import numpy as np
import pandas as pd
import openml

warnings.filterwarnings('ignore')

FAST_TEST = False                 # True = only 4 small datasets, to test that everything works
SEEDS = [0, 1, 2]                 # use [0, 1, 2, 3, 4] for 5 repeats (takes longer)
MODELS = ['logreg', 'rf', 'xgb', 'lgbm', 'mlp']
METRICS = ['acc', 'bacc', 'f1']   # accuracy, balanced accuracy, macro-F1
METRIC_NAMES = {'acc': 'accuracy', 'bacc': 'balanced accuracy', 'f1': 'macro-F1'}
MAIN_MODELS = ['logreg', 'rf', 'xgb']   # the three models of the original paper
FILE_PREFIX = 'ext'

ORIGINAL_IDS = [3, 15, 29, 31, 37, 38, 44, 50, 1049, 1067, 1480,
                11, 23, 46, 54, 307, 12, 14, 16, 22, 28]
NEW_IDS = [18, 182, 188, 458, 469, 1050, 1063, 1068, 1462, 1475, 1487, 1489, 1494, 1497,
           1510, 1464, 4538, 6332, 23381, 40499, 40701, 40966, 40975, 40982, 40983, 40984, 40994]
DATASET_IDS = ORIGINAL_IDS + NEW_IDS
if FAST_TEST:
    ORIGINAL_IDS = ORIGINAL_IDS[:2]
    NEW_IDS = [1063, 1464]
    DATASET_IDS = ORIGINAL_IDS + NEW_IDS
    SEEDS = [0, 1]


def load_dataset(did):
    ds = openml.datasets.get_dataset(did)
    X, y, _, _ = ds.get_data(target=ds.default_target_attribute)
    for col in X.columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            codes = X[col].astype('category').cat.codes.astype(float)
            X[col] = codes.where(codes >= 0, np.nan)
    X = X.astype(float).reset_index(drop=True)
    y = pd.factorize(y)[0]
    return X, y, ds.name

## Part A: data problems, models and measures

The four old problems are unchanged. **New:** `class_noise` (class-dependent label noise): a chosen share of the training labels is moved to the *next* class (class k becomes class k+1, the last class wraps around to the first). For datasets with only two classes this is the same as symmetric noise, so the transfer test is also reported for multi-class datasets alone.

In [ ]:
def inject_missing(X, level, seed=0):
    rng = np.random.default_rng(seed)
    return X.mask(rng.random(X.shape) < level)


def inject_label_noise(y, level, seed=0):
    rng = np.random.default_rng(seed)
    y = y.copy()
    classes = np.unique(y)
    idx = rng.choice(len(y), int(level * len(y)), replace=False)
    for i in idx:
        y[i] = rng.choice(classes[classes != y[i]])
    return y


def inject_class_noise(y, level, seed=0):
    # class-dependent noise: a label moves to the next class (cyclic)
    rng = np.random.default_rng(seed + 12345)
    y = y.copy()
    classes = np.unique(y)
    pos = {c: i for i, c in enumerate(classes)}
    idx = rng.choice(len(y), int(level * len(y)), replace=False)
    for i in idx:
        y[i] = classes[(pos[y[i]] + 1) % len(classes)]
    return y


def inject_duplicates(X, y, level, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), int(level * len(X)), replace=True)
    return pd.concat([X, X.iloc[idx]], ignore_index=True), np.concatenate([y, y[idx]])


def inject_imbalance(X, y, level, seed=0):
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(y, return_counts=True)
    biggest = classes[np.argmax(counts)]
    keep = []
    for c in classes:
        idx = np.where(y == c)[0]
        if c != biggest:
            idx = rng.choice(idx, max(2, int(round(len(idx) * (1 - level)))), replace=False)
        keep.extend(idx)
    keep = np.sort(np.array(keep))
    return X.iloc[keep].reset_index(drop=True), y[keep]


def apply_defect(X_tr, y_tr, defect, level, seed=0):
    if defect == 'missing':
        return inject_missing(X_tr, level, seed), y_tr
    if defect == 'label_noise':
        return X_tr, inject_label_noise(y_tr, level, seed)
    if defect == 'class_noise':
        return X_tr, inject_class_noise(y_tr, level, seed)
    if defect == 'duplicates':
        return inject_duplicates(X_tr, y_tr, level, seed)
    if defect == 'imbalance':
        return inject_imbalance(X_tr, y_tr, level, seed)
    raise ValueError('unknown defect: ' + defect)


DEFECTS = ['missing', 'label_noise', 'class_noise', 'duplicates', 'imbalance']
DEFECT_ORDER = ['label_noise', 'class_noise', 'imbalance', 'duplicates', 'missing']
COMBINED = ['missing', 'label_noise', 'duplicates', 'imbalance']   # as in the original paper
LEVELS = [0.05, 0.1, 0.2, 0.4]
COMBO_LEVELS = [0.1, 0.2]


def apply_defects(X, y, setting, seed=0):
    for d in DEFECT_ORDER:
        if d in setting:
            X, y = apply_defect(X, y, d, setting[d], seed)
    return X, y


def build_settings():
    settings = [{}]
    for d in DEFECTS:
        for lv in LEVELS:
            settings.append({d: lv})
    for lv in COMBO_LEVELS:
        settings.append({d: lv for d in COMBINED})
    return settings


SETTINGS = build_settings()
print('Number of settings per dataset:', len(SETTINGS))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


def make_model(name):
    imputer = SimpleImputer(strategy='median')
    if name == 'logreg':
        return make_pipeline(imputer, StandardScaler(), LogisticRegression(max_iter=1000))
    if name == 'rf':
        return make_pipeline(imputer, RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0))
    if name == 'xgb':
        return make_pipeline(imputer, XGBClassifier(n_estimators=100, n_jobs=-1, random_state=0, verbosity=0))
    if name == 'lgbm':
        return make_pipeline(imputer, LGBMClassifier(n_estimators=100, n_jobs=-1, random_state=0, verbose=-1))
    if name == 'mlp':
        return make_pipeline(imputer, StandardScaler(), MLPClassifier(
            hidden_layer_sizes=(64,), max_iter=200, early_stopping=True, random_state=0))
    raise ValueError('unknown model: ' + name)


def fit_predict(name, X_fit, y_fit, X_eval):
    # trains a model and predicts; the neural network falls back to a version without
    # early stopping if the internal validation split fails (tiny classes)
    try:
        return make_model(name).fit(X_fit, y_fit).predict(X_eval)
    except Exception:
        if name != 'mlp':
            raise
        model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), MLPClassifier(
            hidden_layer_sizes=(64,), max_iter=200, early_stopping=False, random_state=0))
        return model.fit(X_fit, y_fit).predict(X_eval)


def scores(y_true, y_pred):
    return {'acc': accuracy_score(y_true, y_pred),
            'bacc': balanced_accuracy_score(y_true, y_pred),
            'f1': f1_score(y_true, y_pred, average='macro')}

## Part B: meta-features

Unchanged. They are measured on the damaged training data, before any model is trained.

In [ ]:
from scipy.stats import entropy
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

META_COLS = ['n_rows', 'n_features', 'n_classes', 'rows_per_feature',
             'missing_rate', 'duplicate_rate', 'imbalance_ratio', 'outlier_rate',
             'mean_skew', 'mean_kurtosis', 'mean_abs_corr',
             'class_entropy', 'mean_mutual_info',
             'lm_stump', 'lm_tree3', 'lm_nb']


def compute_meta_features(X, y, seed=0):
    rng = np.random.default_rng(seed)
    n, p = X.shape
    classes, counts = np.unique(y, return_counts=True)
    prop = counts / counts.sum()
    mf = {'n_rows': n, 'n_features': p, 'n_classes': len(classes), 'rows_per_feature': n / p,
          'missing_rate': float(X.isna().to_numpy().mean()),
          'duplicate_rate': float(X.assign(_y=y).duplicated().mean()),
          'imbalance_ratio': float(counts.min() / counts.max()),
          'class_entropy': float(entropy(prop) / np.log(len(classes))) if len(classes) > 1 else 0.0}
    A = X.to_numpy(dtype=float)
    mu = np.nanmean(A, axis=0)
    sd = np.nanstd(A, axis=0)
    sd[sd == 0] = np.nan
    z = np.abs((A - mu) / sd)
    valid = ~np.isnan(z)
    mf['outlier_rate'] = float((z[valid] > 3).mean()) if valid.any() else 0.0
    mf['mean_skew'] = float(X.skew().mean())
    mf['mean_kurtosis'] = float(X.kurt().mean())
    if p > 1:
        C = X.corr().abs().to_numpy()
        mf['mean_abs_corr'] = float(np.nanmean(C[np.triu_indices(p, k=1)]))
    else:
        mf['mean_abs_corr'] = 0.0
    m = min(n, 1000)
    idx = rng.choice(n, m, replace=False)
    Xs = X.iloc[idx]
    ys = y[idx]
    Xi = Xs.fillna(Xs.median()).fillna(0.0).to_numpy()
    cols = rng.choice(p, min(p, 20), replace=False)
    try:
        mf['mean_mutual_info'] = float(mutual_info_classif(Xi[:, cols], ys, random_state=seed).mean())
    except Exception:
        mf['mean_mutual_info'] = np.nan

    def landmark(model):
        try:
            return float(cross_val_score(model, Xi, ys, cv=3).mean())
        except Exception:
            return np.nan

    mf['lm_stump'] = landmark(DecisionTreeClassifier(max_depth=1, random_state=0))
    mf['lm_tree3'] = landmark(DecisionTreeClassifier(max_depth=3, random_state=0))
    mf['lm_nb'] = landmark(GaussianNB())
    return mf

## Part A (continued): run the experiments

One file is saved per dataset and seed. If you stop the notebook, run this cell again and it continues where it stopped.

In [ ]:
def run_dataset(did, settings, seeds=(0,), models=MODELS):
    X, y, name = load_dataset(did)
    rows = []
    for seed in seeds:
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
        X_tr = X_tr.reset_index(drop=True)
        clean = {m: scores(y_te, fit_predict(m, X_tr, y_tr, X_te)) for m in models}
        for sid, setting in enumerate(settings):
            Xd, yd = apply_defects(X_tr, y_tr, setting, seed)
            mf = compute_meta_features(Xd, yd, seed)
            for m in models:
                sc = clean[m] if not setting else scores(y_te, fit_predict(m, Xd, yd, X_te))
                row = {'dataset': name, 'dataset_id': did, 'seed': seed, 'setting_id': sid, 'model': m}
                for d in DEFECTS:
                    row['inj_' + d] = setting.get(d, 0.0)
                row.update(mf)
                for k in METRICS:
                    row['clean_' + k] = clean[m][k]
                    row['deg_' + k] = clean[m][k] - sc[k]
                rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
failed = []
for did in DATASET_IDS:
    for seed in SEEDS:
        path = os.path.join(OUTPUT_DIR, FILE_PREFIX + '_' + str(did) + '_seed' + str(seed) + '.csv')
        if os.path.exists(path):
            continue
        t0 = time.time()
        try:
            df = run_dataset(did, SETTINGS, seeds=[seed])
        except Exception as e:
            print('FAILED dataset', did, ':', repr(e)[:200])
            failed.append(did)
            break
        df.to_csv(path, index=False)
        print(f'done {did} ({df["dataset"].iloc[0]}), seed {seed}, in {(time.time() - t0) / 60:.1f} min')
print('Failed datasets:', failed)

## Part C: prepare the results

Seeds are averaged, clean rows are removed, and a dataset is used only if all seeds finished.

In [ ]:
files = sorted(glob.glob(os.path.join(OUTPUT_DIR, FILE_PREFIX + '_*_seed*.csv')))
all_df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
all_df = all_df[all_df['dataset_id'].isin(DATASET_IDS) & all_df['seed'].isin(SEEDS)]
complete = all_df.groupby('dataset_id')['seed'].nunique()
all_df = all_df[all_df['dataset_id'].isin(complete[complete == len(SEEDS)].index)]
print('Datasets:', all_df['dataset_id'].nunique(), '| Seeds:', sorted(all_df['seed'].unique().tolist()))

agg_cols = META_COLS + ['deg_' + k for k in METRICS] + ['inj_' + d for d in DEFECTS]
data = all_df.groupby(['dataset', 'dataset_id', 'setting_id', 'model'], as_index=False)[agg_cols].mean()
data = data[data['setting_id'] != 0].reset_index(drop=True)


def defect_type(row):
    active = [d for d in DEFECTS if row['inj_' + d] > 0]
    return active[0] if len(active) == 1 else 'combined'


data['defect_type'] = data.apply(defect_type, axis=1)
data['level'] = [r['inj_' + r['defect_type']] if r['defect_type'] != 'combined' else r['inj_missing']
                 for _, r in data.iterrows()]
print('Rows used for scoring:', len(data))

## Results 1: how much does each defect cost, for each measure? (improvement 1)

If class imbalance looks harmless for accuracy but not for balanced accuracy or macro-F1, the paper must say so.

In [ ]:
rows = []
for k in METRICS:
    t = data.groupby(['model', 'defect_type'])['deg_' + k].mean().unstack('defect_type')
    t.insert(0, 'measure', METRIC_NAMES[k])
    rows.append(t.reset_index())
loss_table = pd.concat(rows, ignore_index=True)
loss_table = loss_table[['measure', 'model'] + [c for c in loss_table.columns if c not in ('measure', 'model')]].round(3)
loss_table.to_csv(os.path.join(OUTPUT_DIR, 'result1_loss_by_defect_and_measure.csv'), index=False)
loss_table

## Meta-models and helper functions

The same meta-model as in the paper (random forest on the 7 "small set" features).

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr, wilcoxon
from sklearn.linear_model import LinearRegression

SMALL = ['missing_rate', 'duplicate_rate', 'imbalance_ratio', 'class_entropy', 'lm_stump', 'lm_tree3', 'lm_nb']
LM = ['lm_stump', 'lm_tree3', 'lm_nb']


def make_meta(cols_kind='small'):
    return make_pipeline(SimpleImputer(strategy='median', keep_empty_features=True),
                         RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.5,
                                               n_jobs=-1, random_state=0))


def per_ds(y_true, y_pred, ids):
    return pd.Series(np.abs(np.asarray(y_true) - np.asarray(y_pred))).groupby(np.asarray(ids)).mean()


def wil(a, b, alt='less'):
    try:
        return wilcoxon(a, b, alternative=alt).pvalue
    except Exception:
        return np.nan


def holm(pvals):
    # Holm step-down correction; returns adjusted p-values in the original order
    p = np.asarray(pvals, dtype=float)
    out = np.full(len(p), np.nan)
    ok = ~np.isnan(p)
    idx = np.where(ok)[0][np.argsort(p[ok])]
    m = len(idx)
    run = 0.0
    for rank, i in enumerate(idx):
        run = max(run, (m - rank) * p[i])
        out[i] = min(1.0, run)
    return out


def boot_ci(err_meta, err_base, n=2000, seed=0):
    # 95% bootstrap interval (resampling datasets) of the relative error reduction of the meta-model
    rng = np.random.default_rng(seed)
    a, b = np.asarray(err_meta), np.asarray(err_base)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(a), len(a))
        vals.append(1 - a[i].mean() / b[i].mean())
    return np.percentile(vals, [2.5, 97.5])

## Results 2: leave-one-dataset-out on all datasets, for every measure and model (improvements 1 and 3)

For each measure and model: average-drop baseline against the random forest meta-model. `p_holm` is corrected for all comparisons in this table.

In [ ]:
rows = []
for k in METRICS:
    for m in MODELS:
        d = data[data['model'] == m].reset_index(drop=True)
        y = d['deg_' + k].to_numpy()
        pred_avg = np.zeros(len(d))
        pred_meta = np.zeros(len(d))
        for tr, te in LeaveOneGroupOut().split(d, y, d['dataset_id'].to_numpy()):
            pred_avg[te] = y[tr].mean()
            pred_meta[te] = make_meta().fit(d.iloc[tr][SMALL], y[tr]).predict(d.iloc[te][SMALL])
        e_avg = per_ds(y, pred_avg, d['dataset_id'])
        e_meta = per_ds(y, pred_meta, d['dataset_id'])
        lo, hi = boot_ci(e_meta, e_avg)
        rows.append({'measure': METRIC_NAMES[k], 'model': m, 'MAE_avg_drop': np.abs(y - pred_avg).mean(),
                     'MAE_meta': np.abs(y - pred_meta).mean(),
                     'reduction_%': 100 * (1 - e_meta.mean() / e_avg.mean()),
                     'CI95_low_%': 100 * lo, 'CI95_high_%': 100 * hi,
                     'better_on': str(int((e_meta < e_avg).sum())) + '/' + str(len(e_avg)),
                     'p_one_sided': wil(e_meta, e_avg), 'rank_corr': spearmanr(y, pred_meta)[0]})
res2 = pd.DataFrame(rows)
res2['p_holm'] = holm(res2['p_one_sided'].to_numpy())
res2 = res2.round(4)
res2.to_csv(os.path.join(OUTPUT_DIR, 'result2_leave_one_dataset_out.csv'), index=False)
res2

## Results 3: the planned test, now with 5 models, 3 measures and the "told the noise level" baseline (improvements 1, 3 and 6)

Train on Round-1 datasets (all problem types except class-dependent noise), test on the label-noise cases of the Round-2 datasets.
`reduction_%` is the error reduction against the average-drop baseline with a bootstrap 95% interval. The "told the noise level" baseline predicts the average loss seen in training at the true noise level.

In [ ]:
rows = []
store = {}
for k in METRICS:
    for m in MODELS:
        tr = data[(data['model'] == m) & data['dataset_id'].isin(ORIGINAL_IDS) & (data['defect_type'] != 'class_noise')]
        te = data[(data['model'] == m) & ~data['dataset_id'].isin(ORIGINAL_IDS) & (data['defect_type'] == 'label_noise')].reset_index(drop=True)
        y_tr, y_te, ids = tr['deg_' + k].to_numpy(), te['deg_' + k].to_numpy(), te['dataset_id'].to_numpy()
        p_avg = np.full(len(te), y_tr.mean())
        p_meta = make_meta().fit(tr[SMALL], y_tr).predict(te[SMALL])
        trl = tr[tr['defect_type'] == 'label_noise']
        tab = trl.groupby('level')['deg_' + k].mean()
        p_told = np.array([tab.iloc[(tab.index - l).to_numpy().__abs__().argmin()] for l in te['level']])
        e = {n: per_ds(y_te, p, ids) for n, p in [('avg', p_avg), ('meta', p_meta), ('told', p_told)]}
        store[(k, m)] = (te, e)
        lo, hi = boot_ci(e['meta'], e['avg'])
        rows.append({'measure': METRIC_NAMES[k], 'model': m,
                     'MAE_avg_drop': np.abs(y_te - p_avg).mean(), 'MAE_meta': np.abs(y_te - p_meta).mean(),
                     'MAE_told_noise_level': np.abs(y_te - p_told).mean(),
                     'reduction_%': 100 * (1 - e['meta'].mean() / e['avg'].mean()),
                     'CI95_low_%': 100 * lo, 'CI95_high_%': 100 * hi,
                     'better_on': str(int((e['meta'] < e['avg']).sum())) + '/' + str(len(e['avg'])),
                     'p_one_sided': wil(e['meta'], e['avg']),
                     'rank_corr': spearmanr(y_te, p_meta)[0]})
res3 = pd.DataFrame(rows)
res3['p_holm'] = holm(res3['p_one_sided'].to_numpy())
res3 = res3.round(4)
res3.to_csv(os.path.join(OUTPUT_DIR, 'result3_planned_test.csv'), index=False)
res3

## Results 4: transfer to class-dependent noise (improvement 2)

The meta-model is trained **without any class-dependent noise** and tested on the class-dependent-noise cases of the Round-2 datasets. This tests whether it still detects damage when the noise is no longer random. The second table uses only multi-class datasets, because for two classes both noise types are the same.

In [ ]:
rows = []
for k in METRICS:
    for m in MODELS:
        for subset in ['all datasets', 'multi-class only']:
            tr = data[(data['model'] == m) & data['dataset_id'].isin(ORIGINAL_IDS) & (data['defect_type'] != 'class_noise')]
            te = data[(data['model'] == m) & ~data['dataset_id'].isin(ORIGINAL_IDS) & (data['defect_type'] == 'class_noise')]
            if subset == 'multi-class only':
                te = te[te['n_classes'] > 2]
            te = te.reset_index(drop=True)
            if te['dataset_id'].nunique() < 2:
                continue
            y_tr, y_te, ids = tr['deg_' + k].to_numpy(), te['deg_' + k].to_numpy(), te['dataset_id'].to_numpy()
            p_avg = np.full(len(te), y_tr.mean())
            p_meta = make_meta().fit(tr[SMALL], y_tr).predict(te[SMALL])
            e_a, e_m = per_ds(y_te, p_avg, ids), per_ds(y_te, p_meta, ids)
            lo, hi = boot_ci(e_m, e_a)
            rows.append({'measure': METRIC_NAMES[k], 'model': m, 'datasets': subset, 'n_datasets': len(e_a),
                         'MAE_avg_drop': np.abs(y_te - p_avg).mean(), 'MAE_meta': np.abs(y_te - p_meta).mean(),
                         'reduction_%': 100 * (1 - e_m.mean() / e_a.mean()),
                         'CI95_low_%': 100 * lo, 'CI95_high_%': 100 * hi,
                         'better_on': str(int((e_m < e_a).sum())) + '/' + str(len(e_a)),
                         'p_one_sided': wil(e_m, e_a), 'rank_corr': spearmanr(y_te, p_meta)[0]})
res4 = pd.DataFrame(rows)
if len(res4):
    res4['p_holm'] = holm(res4['p_one_sided'].to_numpy())
    res4 = res4.round(4)
else:
    print('Not enough datasets with class-dependent noise to run the transfer test.')
res4.to_csv(os.path.join(OUTPUT_DIR, 'result4_transfer_class_noise.csv'), index=False)
res4

## Results 5: where does the meta-model fail? (improvement 5)

For the planned test (label noise, accuracy), the datasets with the biggest error increase against the baseline, and whether the increase is related to simple dataset properties.

In [ ]:
fail_rows = []
for m in MAIN_MODELS:
    te, e = store[('acc', m)]
    diff = (e['meta'] - e['avg']).rename('error_increase')
    props = te.groupby('dataset_id')[['n_rows', 'n_features', 'n_classes', 'rows_per_feature', 'imbalance_ratio']].mean()
    names = te.groupby('dataset_id')['dataset'].first()
    t = pd.concat([names, diff, e['avg'].rename('MAE_avg_drop'), e['meta'].rename('MAE_meta'), props], axis=1)
    t.insert(0, 'model', m)
    fail_rows.append(t.sort_values('error_increase', ascending=False).head(5).reset_index())
    print(m, '- relation between error increase and dataset properties (Spearman):')
    print({c: round(spearmanr(diff, props.loc[diff.index, c])[0], 2) for c in props.columns})
failures = pd.concat(fail_rows, ignore_index=True).round(4)
failures.to_csv(os.path.join(OUTPUT_DIR, 'result5_failures.csv'), index=False)
failures

## Results 6: ablation with all 5 models (optional, same idea as the earlier extra check)

Does the gain survive without the landmarking features?

In [ ]:
rows = []
for m in MODELS:
    tr = data[(data['model'] == m) & data['dataset_id'].isin(ORIGINAL_IDS) & (data['defect_type'] != 'class_noise')]
    te = data[(data['model'] == m) & ~data['dataset_id'].isin(ORIGINAL_IDS) & (data['defect_type'] == 'label_noise')].reset_index(drop=True)
    y_tr, y_te, ids = tr['deg_acc'].to_numpy(), te['deg_acc'].to_numpy(), te['dataset_id'].to_numpy()
    base = per_ds(y_te, np.full(len(te), y_tr.mean()), ids)
    for name, cols in [('small set', SMALL), ('small set without landmarking', [c for c in SMALL if c not in LM]),
                       ('landmarking only', LM)]:
        p = make_meta().fit(tr[cols], y_tr).predict(te[cols])
        e = per_ds(y_te, p, ids)
        rows.append({'model': m, 'features': name, 'MAE': np.abs(y_te - p).mean(),
                     'MAE_avg_drop': np.abs(y_te - y_tr.mean()).mean(),
                     'reduction_%': 100 * (1 - e.mean() / base.mean()),
                     'better_on': str(int((e < base).sum())) + '/' + str(len(e)), 'p_one_sided': wil(e, base)})
res6 = pd.DataFrame(rows).round(4)
res6.to_csv(os.path.join(OUTPUT_DIR, 'result6_ablation.csv'), index=False)
res6

## Part D: a more realistic noise type: instance-dependent label noise (improvement 4)

I could not find a tabular benchmark with **verified real label errors** and known ground truth, so this part uses the closest realistic substitute. In real data, labels are wrong mostly for examples that are hard or ambiguous. Here, a model first estimates how uncertain each training example is (out-of-bag probabilities of a random forest). Examples are then chosen for flipping **with probability proportional to their uncertainty**, and each chosen label is moved to the class the model finds most likely besides the true one. The share of flipped labels is the same as for the other noise types (5, 10, 20, 40%).

This is still synthetic noise, but it is feature-dependent, so it is much harder than random noise. The paper should call it "instance-dependent noise", not "real noise".

It runs only on the Round-2 datasets (never used to train the meta-model) and takes much less time than the main experiment.

In [ ]:
def inject_instance_noise(X, y, level, seed=0):
    # label noise that depends on the example: uncertain examples are flipped more often
    rng = np.random.default_rng(seed + 777)
    y = np.asarray(y)
    Xi = X.fillna(X.median()).fillna(0.0).to_numpy()
    rf = RandomForestClassifier(n_estimators=100, oob_score=True, n_jobs=-1, random_state=seed).fit(Xi, y)
    proba = rf.oob_decision_function_
    proba = np.where(np.isnan(proba), 1.0 / proba.shape[1], proba)
    classes = rf.classes_
    pos = {c: i for i, c in enumerate(classes)}
    srt = np.sort(proba, axis=1)
    margin = srt[:, -1] - srt[:, -2] if proba.shape[1] > 1 else np.ones(len(y))
    weight = (1.0 - margin) + 1e-3
    n_flip = int(level * len(y))
    idx = rng.choice(len(y), n_flip, replace=False, p=weight / weight.sum())
    y2 = y.copy()
    for i in idx:
        pr = proba[i].copy()
        pr[pos[y[i]]] = -1.0
        y2[i] = classes[int(np.argmax(pr))]
    return y2


def run_instance_noise(did, seeds=(0,), models=MODELS):
    X, y, name = load_dataset(did)
    rows = []
    for seed in seeds:
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
        X_tr = X_tr.reset_index(drop=True)
        clean = {m: scores(y_te, fit_predict(m, X_tr, y_tr, X_te)) for m in models}
        for lv in LEVELS:
            yd = inject_instance_noise(X_tr, y_tr, lv, seed)
            mf = compute_meta_features(X_tr, yd, seed)
            for m in models:
                sc = scores(y_te, fit_predict(m, X_tr, yd, X_te))
                row = {'dataset': name, 'dataset_id': did, 'seed': seed, 'level': lv, 'model': m}
                row.update(mf)
                for k in METRICS:
                    row['deg_' + k] = clean[m][k] - sc[k]
                rows.append(row)
    return pd.DataFrame(rows)


inst_files = []
for did in NEW_IDS:
    for seed in SEEDS:
        path = os.path.join(OUTPUT_DIR, 'inst_' + str(did) + '_seed' + str(seed) + '.csv')
        if not os.path.exists(path):
            t0 = time.time()
            try:
                df = run_instance_noise(did, seeds=[seed])
            except Exception as e:
                print('FAILED dataset', did, ':', repr(e)[:200])
                continue
            df.to_csv(path, index=False)
            print(f'done {did} ({df["dataset"].iloc[0]}), seed {seed}, in {(time.time() - t0) / 60:.1f} min')
        inst_files.append(path)
print('Finished instance-noise files:', len(inst_files))

In [ ]:
inst = pd.concat([pd.read_csv(f) for f in inst_files if os.path.exists(f)], ignore_index=True)
inst = inst.groupby(['dataset', 'dataset_id', 'level', 'model'], as_index=False)[META_COLS + ['deg_' + k for k in METRICS]].mean()

rows = []
for k in METRICS:
    for m in MODELS:
        tr = data[(data['model'] == m) & data['dataset_id'].isin(ORIGINAL_IDS) & (data['defect_type'] != 'class_noise')]
        te = inst[inst['model'] == m].reset_index(drop=True)
        if te['dataset_id'].nunique() < 2:
            continue
        y_tr, y_te, ids = tr['deg_' + k].to_numpy(), te['deg_' + k].to_numpy(), te['dataset_id'].to_numpy()
        p_avg = np.full(len(te), y_tr.mean())
        p_meta = make_meta().fit(tr[SMALL], y_tr).predict(te[SMALL])
        trl = tr[tr['defect_type'] == 'label_noise']
        tab = trl.groupby('level')['deg_' + k].mean()
        p_told = np.array([tab.iloc[(tab.index - l).to_numpy().__abs__().argmin()] for l in te['level']])
        e_a, e_m, e_t = per_ds(y_te, p_avg, ids), per_ds(y_te, p_meta, ids), per_ds(y_te, p_told, ids)
        lo, hi = boot_ci(e_m, e_a)
        rows.append({'measure': METRIC_NAMES[k], 'model': m, 'n_datasets': len(e_a),
                     'real_mean_loss': y_te.mean(),
                     'MAE_avg_drop': np.abs(y_te - p_avg).mean(), 'MAE_meta': np.abs(y_te - p_meta).mean(),
                     'MAE_told_noise_level': np.abs(y_te - p_told).mean(),
                     'reduction_%': 100 * (1 - e_m.mean() / e_a.mean()),
                     'CI95_low_%': 100 * lo, 'CI95_high_%': 100 * hi,
                     'better_on': str(int((e_m < e_a).sum())) + '/' + str(len(e_a)),
                     'p_one_sided': wil(e_m, e_a), 'rank_corr': spearmanr(y_te, p_meta)[0]})
res7 = pd.DataFrame(rows)
if len(res7):
    res7['p_holm'] = holm(res7['p_one_sided'].to_numpy())
    res7 = res7.round(4)
    res7.to_csv(os.path.join(OUTPUT_DIR, 'result7_instance_dependent_noise.csv'), index=False)
res7

## Done

Send the seven result tables (or the `result*.csv` files from the Drive folder `data_quality_paper_extended`) back, and the paper can be updated with them.